<a href="https://colab.research.google.com/github/emankhalid019/sql-from-scratch/blob/main/Day04_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Theory + Commands
# SQL TUTORIAL - PART 5: Primary Key, Foreign Key, Check, Default, Create Index, Auto Increment, Dates, Views, SQL Injection, Parameters, Prepared Statements, Hosting


import sqlite3
import pandas as pd
import datetime

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

def run(query, params=None):
    """Executes a query (with optional parameters). Returns a table
    if it's a SELECT, else a status message."""
    if params:
        cur.execute(query, params)
    else:
        cur.execute(query)
    conn.commit()
    if query.strip().upper().startswith("SELECT"):
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    return "Query executed successfully."






## SQL PRIMARY KEY
# A PRIMARY KEY uniquely identifies each row in a table.
# - Must contain UNIQUE values
# - Cannot contain NULL values
# - A table can have only ONE primary key (which may be one or more columns -- a "composite" primary key)
print(run("""
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT NOT NULL
);
"""))

run("INSERT INTO Departments VALUES (1, 'IT');")
run("INSERT INTO Departments VALUES (2, 'HR');")
print(run("SELECT * FROM Departments;"))

# This would FAIL -- DepartmentID 1 already exists (violates PRIMARY KEY):
try:
    run("INSERT INTO Departments VALUES (1, 'Finance');")
except Exception as e:
    print("PRIMARY KEY constraint blocked this insert:", e)









## SQL FOREIGN KEY
# A FOREIGN KEY links a column in one table to the PRIMARY KEY ofanother table. It prevents actions that would break the link between the two tables (referential integrity).
run("PRAGMA foreign_keys = ON;")  # SQLite requires this to enforce FKs

print(run("""
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    DepartmentID INTEGER,
    FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID)
);
"""))

run("INSERT INTO Employees VALUES (1, 'Ali', 1);")   # valid, DepartmentID 1 exists
print(run("SELECT * FROM Employees;"))

# This would FAIL -- DepartmentID 99 does not exist in Departments:
try:
    run("INSERT INTO Employees VALUES (2, 'Sara', 99);")
except Exception as e:
    print("FOREIGN KEY constraint blocked this insert:", e)










## SQL CHECK
# CHECK ensures that all values in a column satisfy a specific condition.
print(run("""
CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    Price INTEGER CHECK (Price > 0)
);
"""))

run("INSERT INTO Products VALUES (1, 'Laptop', 90000);")
print(run("SELECT * FROM Products;"))

# This would FAIL -- Price must be greater than 0:
try:
    run("INSERT INTO Products VALUES (2, 'Broken Item', -100);")
except Exception as e:
    print("CHECK constraint blocked this insert:", e)










## SQL DEFAULT
# DEFAULT sets an automatic value for a column when no value is provided during INSERT.

print(run("""
CREATE TABLE Orders (
    OrderID INTEGER PRIMARY KEY,
    ProductName TEXT,
    OrderDate TEXT DEFAULT (DATE('now')),
    Status TEXT DEFAULT 'Pending'
);
"""))
run("INSERT INTO Orders (OrderID, ProductName) VALUES (1, 'Laptop');")
print(run("SELECT * FROM Orders;"))# OrderDate and Status were filled in automatically










## SQL CREATE INDEX
# An INDEX speeds up searches/queries on a table's column(s).
# It does NOT change the data, only how fast the database can find it.
# Note: indexes are invisible to query results , they're a performance tool, not something you SELECT from.

run("CREATE INDEX idx_employee_name ON Employees (Name);")
print("Index 'idx_employee_name' created on Employees(Name).")

# UNIQUE index -- also enforces uniqueness like a UNIQUE constraint:
run("CREATE UNIQUE INDEX idx_department_name ON Departments (DepartmentName);")
print("Unique index created on Departments(DepartmentName).")

# Dropping an index:
#   DROP INDEX idx_employee_name;








## SQL AUTO INCREMENT
# AUTO INCREMENT automatically generates a unique number for a new row's primary key, so you don't have to specify it manually.
# In SQLite, an INTEGER PRIMARY KEY column auto-increments by default.

print(run("""
CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL
);
"""))
run("INSERT INTO Customers (Name) VALUES ('Zara');")   # CustomerID auto-generated
run("INSERT INTO Customers (Name) VALUES ('Bilal');")
print(run("SELECT * FROM Customers;"))

# Standard syntax in other databases:
#   MySQL:      CustomerID INT AUTO_INCREMENT PRIMARY KEY
#   SQL Server: CustomerID INT IDENTITY(1,1) PRIMARY KEY








## SQL DATES
# SQL has special functions/types for working with dates and times.
# Common SQLite date functions: DATE('now'), TIME('now'), DATETIME('now')

print(run("SELECT DATE('now') AS Today;"))
print(run("SELECT DATETIME('now') AS Now;"))

run("INSERT INTO Orders (OrderID, ProductName, OrderDate) VALUES (2, 'Mouse', '2026-01-15');")
print(run("SELECT * FROM Orders WHERE OrderDate < '2026-06-01';"))

# Standard date data types in other databases:
#   MySQL:      DATE, DATETIME, TIMESTAMP, TIME, YEAR
#   SQL Server: DATE, DATETIME, DATETIME2, SMALLDATETIME, TIME









## SQL VIEWS
# A VIEW is a virtual table based on the result of a SELECT query.
# It doesn't store data itself -- it always reflects the current data in the underlying table(s).
run("""
CREATE VIEW HighValueOrders AS
SELECT OrderID, ProductName, Status
FROM Orders
WHERE Status = 'Pending';
""")

print(run("SELECT * FROM HighValueOrders;"))

# Views are queried exactly like normal tables:
print(run("SELECT ProductName FROM HighValueOrders;"))

# Dropping a view:
#   DROP VIEW HighValueOrders;







## SQL INJECTION
# SQL Injection is a code-injection technique where malicious SQL is inserted into an input field to manipulate or damage a database.
# It happens when user input is directly concatenated into a querystring instead of being safely passed as a parameter.

# DANGEROUS pattern (never do this):
#   user_input = "Ali' OR '1'='1"
#   query = "SELECT * FROM Employees WHERE Name = '" + user_input + "'"
#   -> becomes: SELECT * FROM Employees WHERE Name = 'Ali' OR '1'='1'
#   -> '1'='1' is always true, so it returns ALL rows, bypassing the filter!

unsafe_input = "Ali' OR '1'='1"
unsafe_query = "SELECT * FROM Employees WHERE Name = '" + unsafe_input + "'"
print("Unsafe query built from user input:", unsafe_query)
print(run(unsafe_query))
# Notice this returns every employee, not just one named "Ali' OR '1'='1"
# this is exactly how SQL injection attacks work.

# The fix is always to use PARAMETERS (shown next), never stringconcatenation of user input into SQL.





## SQL PARAMETERS
# Parameters (placeholders) let you safely pass values into a query.
# The database treats the value strictly as DATA, never as executable
# SQL code this prevents SQL injection completely.
safe_input = "Ali' OR '1'='1"   # same malicious-looking input as before
print(run("SELECT * FROM Employees WHERE Name = ?;", (safe_input,)))
# Returns NO rows, because the whole string is treated as a literal name to search for -- the injection attempt is neutralized.

print(run("SELECT * FROM Employees WHERE Name = ?;", ("Ali",)))# This works correctly and returns Ali's row.







## SQL PREPARED STATEMENTS
# A prepared statement is a query template that the database compiles ONCE, then executes many times with different parameter values.
# It's both a security best practice (prevents SQL injection) and a performance optimization for repeated queries.
prepared_query = "INSERT INTO Customers (Name) VALUES (?);"
new_customers = [("Hamza",), ("Fatima",), ("Noor",)]

cur.executemany(prepared_query, new_customers)
conn.commit()
print(run("SELECT * FROM Customers;"))








## SQL HOSTING
# SQL Hosting refers to where your database physically runs, e.g.:
# - A cloud provider's managed database service (AWS RDS, Azure SQL,Google Cloud SQL, PlanetScale, Supabase, etc.)
# - A traditional web host's database (often MySQL via cPanel)
# - Your own server (self-hosted MySQL/PostgreSQL/SQL Server instance)
# This is a conceptual/infrastructure topic rather than a SQL command there's no SQL syntax for "hosting" itself. Colab's SQLite databaseused in this tutorial is
#temporary and local (in-memory or a file),not a hosted production database.


conn.close()







# Keyword               Purpose                                  Example

# PRIMARY KEY            Uniquely identifies each row              ID INTEGER PRIMARY KEY
# FOREIGN KEY             Links to another table's primary key      FOREIGN KEY (DeptID) REFERENCES Dept(ID)
# CHECK                   Value must satisfy a condition             Price INTEGER CHECK (Price > 0)
# DEFAULT                 Sets an automatic default value            Status TEXT DEFAULT 'Pending'
# CREATE INDEX            Speeds up searches on a column              CREATE INDEX idx ON T(col);
# AUTO INCREMENT           Auto-generates unique key values           ID INTEGER PRIMARY KEY AUTOINCREMENT
# Dates                   Working with date/time values              DATE('now'), DATETIME('now')
# VIEW                    Virtual table from a saved SELECT          CREATE VIEW v AS SELECT ...
# SQL Injection           Attack via unsanitized user input          Name = '" + input + "'  (never do this)
# Parameters               Safely pass values into a query            WHERE Name = ?
# Prepared Statements      Reusable, precompiled, safe query          executemany(query, [params, ...])
# Hosting                  Where the database physically runs         (concept, not SQL syntax)
